# Testes de k na SVD

Reconstruído em 28/10/2026 a partir da [conversa compartilhada](https://chatgpt.com/share/6ae27680-e994-83e9-a410-8026e2a56e00). Os notebooks locais estavam vazios. O código abaixo corresponde à versão mais recente recuperada; não há resultados de execução preservados.

O notebook seleciona 500 imagens de Cat e Dog, calcula a SVD truncada, salva B e C e compara o MSE para os valores de k informados. Execute as células em ordem a partir da pasta do projeto.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.sparse.linalg import svds
from pathlib import Path

diretorio = Path("dataset/PetImages")
diretorio_saida = Path("fatores_svd")
diretorio_saida.mkdir(exist_ok=True)

# Configure estes três valores diretamente no notebook.
N = 500
k_max = None
passo = None

N_PIORES = 100
if not isinstance(N, int) or N < 1 or N % 2 != 0:
    raise ValueError("N deve ser um inteiro positivo e par para dividir igualmente entre Cat e Dog.")
if N < N_PIORES:
    raise ValueError(f"N deve ser pelo menos {N_PIORES} para selecionar as {N_PIORES} piores imagens.")
n_gatos = N // 2
n_cachorros = N // 2

if not isinstance(k_max, int) or not isinstance(passo, int):
    raise ValueError("Defina k_max e passo como inteiros na configuração acima.")
if k_max < 1 or passo < 1:
    raise ValueError("k máximo e passo devem ser positivos.")
if passo > k_max:
    raise ValueError("O passo não pode ser maior que k máximo.")
valores_k = list(range(passo, k_max + 1, passo))
print(f"Selecionando {n_gatos} gatos e {n_cachorros} cachorros.")
print("Valores de k:", valores_k)

In [ ]:
rng = np.random.default_rng()
imagens_selecionadas = []
for classe, quantidade in [("Cat", n_gatos), ("Dog", n_cachorros)]:
    pasta = diretorio / classe
    arquivos = sorted(
        arquivo for arquivo in pasta.iterdir()
        if arquivo.is_file() and arquivo.suffix.lower() in [".jpg", ".png"]
    )
    if quantidade > len(arquivos):
        raise ValueError(f"Existem apenas {len(arquivos)} imagens em {classe}, mas foram solicitadas {quantidade}.")
    indices = rng.choice(len(arquivos), size=quantidade, replace=False)
    for indice in indices:
        imagens_selecionadas.append((classe, arquivos[indice]))
print("Total de imagens selecionadas:", len(imagens_selecionadas))

In [ ]:
def fatorar_svd(A, k_max):
    if k_max >= min(A.shape):
        raise ValueError(f"k máximo ({k_max}) deve ser menor que {min(A.shape)} para a imagem {A.shape}")
    U, S, Vt = svds(A, k=k_max, which="LM")
    indices = np.argsort(S)[::-1]
    U, S, Vt = U[:, indices], S[indices], Vt[indices, :]
    B = U
    C = S[:, None] * Vt
    return B, C

def reconstruir(B, C, k):
    return B[:, :k] @ C[:k, :]

def calcular_mse(A, A_k):
    return np.mean((A - A_k) ** 2)

In [ ]:
resultados_mse = []
identificadores = []
for indice, (classe, caminho) in enumerate(imagens_selecionadas):
    with Image.open(caminho) as imagem:
        A = np.array(imagem.convert("L"), dtype=float)
    B, C = fatorar_svd(A, k_max)
    pasta_saida = diretorio_saida / classe
    pasta_saida.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(pasta_saida / f"{caminho.name}.npz", vetores=B, coeficientes=C)
    erros = []
    for k in valores_k:
        A_k = np.clip(reconstruir(B, C, k), 0, 255)
        erros.append(calcular_mse(A, A_k))
    resultados_mse.append(erros)
    identificadores.append((classe, caminho.name))
    if (indice + 1) % 25 == 0:
        print(f"Processadas {indice + 1}/{N} imagens")
resultados_mse = np.array(resultados_mse)
print("Dimensão da matriz de MSE:", resultados_mse.shape)

In [ ]:
mse_medio = np.mean(resultados_mse, axis=0)
plt.figure(figsize=(11, 6))
plt.plot(valores_k, mse_medio, marker="o", linewidth=2)
plt.title("MSE médio das 500 imagens")
plt.xlabel("k")
plt.ylabel("MSE médio")
plt.xticks(valores_k)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mse_por_imagem = np.mean(resultados_mse, axis=1)
indices_ordenados = np.argsort(mse_por_imagem)[::-1]
indices_piores = indices_ordenados[:N_PIORES]
mse_medio_piores = np.mean(resultados_mse[indices_piores, :], axis=0)
plt.figure(figsize=(11, 6))
plt.plot(valores_k, mse_medio_piores, marker="o", linewidth=2)
plt.title("MSE médio das 100 imagens com maiores erros médios")
plt.xlabel("k")
plt.ylabel("MSE médio")
plt.xticks(valores_k)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
N_PIORES_VISUALIZAR = 10
indices_10_piores = indices_ordenados[:N_PIORES_VISUALIZAR]
for posicao, indice in enumerate(indices_10_piores):
    classe, caminho = imagens_selecionadas[indice]
    with Image.open(caminho) as imagem:
        A = np.array(imagem.convert("L"), dtype=float)
    arquivo_svd = diretorio_saida / classe / f"{caminho.name}.npz"
    with np.load(arquivo_svd) as dados:
        B, C = dados["vetores"], dados["coeficientes"]
    erros = resultados_mse[indice]
    fig, axes = plt.subplots(1, len(valores_k) + 1, figsize=(3.5 * (len(valores_k) + 1), 4), squeeze=False)
    axes = axes.flatten()
    fig.suptitle(f"{posicao + 1}ª pior imagem — {classe}/{caminho.name}\nMSE médio = {mse_por_imagem[indice]:.2f}", fontsize=16)
    axes[0].imshow(A, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Original")
    axes[0].axis("off")
    for j, k in enumerate(valores_k):
        A_k = np.clip(reconstruir(B, C, k), 0, 255)
        axes[j + 1].imshow(A_k, cmap="gray", vmin=0, vmax=255)
        axes[j + 1].set_title(f"k = {k}\nMSE = {erros[j]:.2f}")
        axes[j + 1].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 22))
axes = axes.flatten()
fig.suptitle("Evolução do MSE das 10 piores imagens", fontsize=20)
for posicao, indice in enumerate(indices_10_piores):
    classe, nome = identificadores[indice]
    erros = resultados_mse[indice]
    ax = axes[posicao]
    ax.plot(valores_k, erros, marker="o", linewidth=2)
    ax.set_title(f"{posicao + 1}ª pior — {classe}/{nome}", fontsize=12)
    ax.set_xlabel("k")
    ax.set_ylabel("MSE")
    ax.set_xticks(valores_k)
    ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
n_valores = []
m_valores = []
for classe, caminho in imagens_selecionadas:
    with Image.open(caminho) as imagem:
        m, n = imagem.size
    n_valores.append(n)
    m_valores.append(m)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(n_valores, edgecolor="black")
axes[0].set_title("Distribuição de n (altura)")
axes[0].set_xlabel("n (pixels)")
axes[0].set_ylabel("Frequência")
axes[0].grid(axis="y", alpha=0.3)
axes[1].hist(m_valores, edgecolor="black")
axes[1].set_title("Distribuição de m (largura)")
axes[1].set_xlabel("m (pixels)")
axes[1].set_ylabel("Frequência")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Proporções das imagens

A proporção de cada imagem é `largura / altura = m / n`. O histograma usa as mesmas imagens selecionadas nas células anteriores; a altura de cada barra mostra quantas imagens têm proporção naquele intervalo.


In [ ]:
proporcoes = np.asarray(m_valores, dtype=float) / np.asarray(n_valores, dtype=float)

plt.figure(figsize=(10, 5))
plt.hist(proporcoes, edgecolor="black")
plt.title("Distribuição das proporções das imagens selecionadas")
plt.xlabel("Proporção (largura / altura)")
plt.ylabel("Número de imagens")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()